# T33 - 信用债规模

## 第6章：可视化与报告

---

### 可视化模块概述

本章节实现规模数据的可视化展示，包括：
1. 规模趋势图
2. 结构分布图
3. 集中度图
4. 增长率图

In [ ]:
# 导入依赖
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.font_manager import FontProperties
import seaborn as sns
import sqlalchemy
from sqlalchemy import text, create_engine
from datetime import datetime, timedelta
import sqlalchemy.pool as pool
import os
import warnings

warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 从环境变量获取配置
MYSQL_HOST = os.environ.get('MYSQL_HOST', 'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com')
MYSQL_PORT = os.environ.get('MYSQL_PORT', '3306')
MYSQL_USER = os.environ.get('MYSQL_USER', 'hz_work')
MYSQL_PASSWORD = os.environ.get('MYSQL_PASSWORD', '')
MYSQL_DATABASE = os.environ.get('MYSQL_DATABASE', 'bond')

# 创建数据库连接
connection_string = f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}'
sql_engine = create_engine(connection_string, poolclass=pool.NullPool)

print("可视化模块初始化完成")

### 数据准备

In [ ]:
# 获取数据
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

# 获取最新日期
with sql_engine.connect() as conn:
    latest_dt = pd.read_sql("SELECT MAX(DT) as max_dt FROM bond.信用债规模", conn)
    latest_date = latest_dt['max_dt'].iloc[0]

print(f"数据日期范围: {start_date} ~ {end_date}")
print(f"最新数据日期: {latest_date}")

### 1. 规模趋势图

In [ ]:
def plot_scale_trend(engine, start_date, end_date, figsize=(14, 8)):
    """
    绘制规模趋势图
    """
    # 获取全部债券规模
    sql = """
    SELECT DT, 余额
    FROM bond.信用债规模
    WHERE 分类方式 = '大类' AND 分类 = '全部'
    AND DT >= :start_date AND DT <= :end_date
    ORDER BY DT
    """
    
    with engine.connect() as conn:
        data = pd.read_sql(text(sql), conn, params={'start_date': start_date, 'end_date': end_date})
    
    if len(data) == 0:
        print("没有数据")
        return
    
    data['DT'] = pd.to_datetime(data['DT'])
    
    # 计算移动平均
    data['MA20'] = data['余额'].rolling(20).mean()
    data['MA60'] = data['余额'].rolling(60).mean()
    
    # 绘图
    fig, ax = plt.subplots(figsize=figsize)
    
    ax.plot(data['DT'], data['余额'], label='日度规模', alpha=0.5, linewidth=0.8)
    ax.plot(data['DT'], data['MA20'], label='20日均线', linewidth=1.5)
    ax.plot(data['DT'], data['MA60'], label='60日均线', linewidth=1.5)
    
    ax.set_xlabel('日期')
    ax.set_ylabel('规模（亿元）')
    ax.set_title('信用债市场总规模趋势')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    # 格式化x轴日期
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    return data

# 绘制规模趋势图
trend_data = plot_scale_trend(sql_engine, start_date, end_date)

### 2. 大类结构分布图

In [ ]:
def plot_major_structure(engine, dt, figsize=(12, 6)):
    """
    绘制大类结构饼图
    """
    sql = """
    SELECT 分类, 余额
    FROM bond.信用债规模
    WHERE DT = :dt AND 分类方式 = '大类' AND 分类 != '全部'
    ORDER BY 余额 DESC
    """
    
    with engine.connect() as conn:
        data = pd.read_sql(text(sql), conn, params={'dt': str(dt)})
    
    if len(data) == 0:
        print("没有数据")
        return
    
    # 绘制饼图
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # 饼图
    colors = plt.cm.Set3(np.linspace(0, 1, len(data)))
    axes[0].pie(data['余额'], labels=data['分类'], autopct='%1.1f%%', 
                colors=colors, startangle=90)
    axes[0].set_title(f'大类规模分布 ({dt})')
    
    # 条形图
    bars = axes[1].barh(data['分类'], data['余额'], color=colors)
    axes[1].set_xlabel('规模（亿元）')
    axes[1].set_title('大类规模对比')
    axes[1].invert_yaxis()
    
    # 添加数值标签
    for bar, value in zip(bars, data['余额']):
        axes[1].text(bar.get_width(), bar.get_y() + bar.get_height()/2, 
                     f'{value:,.0f}', va='center', ha='left')
    
    plt.tight_layout()
    plt.show()
    
    return data

# 绘制大类结构图
major_data = plot_major_structure(sql_engine, latest_date)

### 3. 评级分布图

In [ ]:
def plot_rating_distribution(engine, dt, figsize=(12, 6)):
    """
    绘制评级分布图
    """
    sql = """
    SELECT 子类 as 评级, 余额
    FROM bond.信用债规模
    WHERE DT = :dt AND 分类方式 = '评级' AND 分类 = '全部'
    ORDER BY 余额 DESC
    """
    
    with engine.connect() as conn:
        data = pd.read_sql(text(sql), conn, params={'dt': str(dt)})
    
    if len(data) == 0:
        print("没有数据")
        return
    
    # 定义评级颜色
    rating_colors = {
        'AAA': '#2E86AB',
        'AA+': '#A23B72',
        'AA': '#F18F01',
        'AA-': '#C73E1D',
        'A': '#3B1F2B',
        '其他': '#95A3B3',
        '无评级': '#CCCCCC'
    }
    
    colors = [rating_colors.get(r, '#888888') for r in data['评级']]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    bars = ax.bar(data['评级'], data['余额'], color=colors)
    ax.set_xlabel('评级')
    ax.set_ylabel('规模（亿元）')
    ax.set_title(f'评级分布 ({dt})')
    
    # 添加数值标签
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:,.0f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    return data

# 绘制评级分布图
rating_data = plot_rating_distribution(sql_engine, latest_date)

### 4. 久期分布图

In [ ]:
def plot_term_distribution(engine, dt, figsize=(12, 6)):
    """
    绘制久期分布图
    """
    sql = """
    SELECT CAST(子类 AS DECIMAL) as 久期, 余额
    FROM bond.信用债规模
    WHERE DT = :dt AND 分类方式 = '久期' AND 分类 = '全部'
    ORDER BY 久期
    """
    
    with engine.connect() as conn:
        data = pd.read_sql(text(sql), conn, params={'dt': str(dt)})
    
    if len(data) == 0:
        print("没有数据")
        return
    
    data['久期'] = data['久期'].astype(str) + '年'
    
    fig, ax = plt.subplots(figsize=figsize)
    
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(data)))
    bars = ax.bar(data['久期'], data['余额'], color=colors)
    
    ax.set_xlabel('久期')
    ax.set_ylabel('规模（亿元）')
    ax.set_title(f'久期分布 ({dt})')
    
    # 添加数值标签
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:,.0f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    return data

# 绘制久期分布图
term_data = plot_term_distribution(sql_engine, latest_date)

### 5. 大类趋势对比图

In [ ]:
def plot_major_comparison(engine, start_date, end_date, figsize=(14, 8)):
    """
    绘制大类趋势对比图
    """
    sql = """
    SELECT DT, 分类, 余额
    FROM bond.信用债规模
    WHERE 分类方式 = '大类' AND 分类 != '全部'
    AND DT >= :start_date AND DT <= :end_date
    ORDER BY DT, 分类
    """
    
    with engine.connect() as conn:
        data = pd.read_sql(text(sql), conn, params={'start_date': start_date, 'end_date': end_date})
    
    if len(data) == 0:
        print("没有数据")
        return
    
    data['DT'] = pd.to_datetime(data['DT'])
    
    # 透视表
    pivot_data = data.pivot(index='DT', columns='分类', values='余额')
    
    # 绘图
    fig, ax = plt.subplots(figsize=figsize)
    
    colors = plt.cm.Set1(np.linspace(0, 1, len(pivot_data.columns)))
    
    for i, col in enumerate(pivot_data.columns):
        ax.plot(pivot_data.index, pivot_data[col], label=col, 
                color=colors[i], linewidth=1.5)
    
    ax.set_xlabel('日期')
    ax.set_ylabel('规模（亿元）')
    ax.set_title('各大类规模趋势对比')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    # 格式化x轴日期
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    return pivot_data

# 绘制大类趋势对比图
comparison_data = plot_major_comparison(sql_engine, start_date, end_date)

### 6. 增长率热力图

In [ ]:
def plot_growth_heatmap(engine, start_date, end_date, figsize=(14, 8)):
    """
    绘制增长率热力图
    """
    sql = """
    SELECT DT, 分类, 余额
    FROM bond.信用债规模
    WHERE 分类方式 = '大类' AND 分类 != '全部'
    AND DT >= :start_date AND DT <= :end_date
    ORDER BY DT, 分类
    """
    
    with engine.connect() as conn:
        data = pd.read_sql(text(sql), conn, params={'start_date': start_date, 'end_date': end_date})
    
    if len(data) == 0:
        print("没有数据")
        return
    
    data['DT'] = pd.to_datetime(data['DT'])
    
    # 透视表
    pivot_data = data.pivot(index='DT', columns='分类', values='余额')
    
    # 计算月度增长率
    pivot_data_monthly = pivot_data.resample('M').last()
    growth_rate = pivot_data_monthly.pct_change() * 100
    growth_rate = growth_rate.dropna()
    
    # 绘制热力图
    fig, ax = plt.subplots(figsize=figsize)
    
    sns.heatmap(growth_rate.T, annot=True, fmt='.2f', cmap='RdYlGn', 
                center=0, ax=ax, cbar_kws={'label': '增长率(%)'})
    
    ax.set_title('各大类月度规模增长率热力图')
    ax.set_xlabel('月份')
    ax.set_ylabel('分类')
    
    plt.tight_layout()
    plt.show()
    
    return growth_rate

# 绘制增长率热力图
heatmap_data = plot_growth_heatmap(sql_engine, start_date, end_date)

### 7. 综合仪表板

In [ ]:
def create_dashboard(engine, dt, start_date, end_date, figsize=(16, 12)):
    """
    创建综合仪表板
    """
    fig = plt.figure(figsize=figsize)
    
    # 2x2布局
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
    
    # 1. 规模趋势
    ax1 = fig.add_subplot(gs[0, 0])
    sql_trend = """
    SELECT DT, 余额
    FROM bond.信用债规模
    WHERE 分类方式 = '大类' AND 分类 = '全部'
    AND DT >= :start_date AND DT <= :end_date
    ORDER BY DT
    """
    with engine.connect() as conn:
        trend = pd.read_sql(text(sql_trend), conn, params={'start_date': start_date, 'end_date': end_date})
    
    if len(trend) > 0:
        trend['DT'] = pd.to_datetime(trend['DT'])
        trend['MA20'] = trend['余额'].rolling(20).mean()
        ax1.plot(trend['DT'], trend['余额'], label='日度规模', alpha=0.5, linewidth=0.8)
        ax1.plot(trend['DT'], trend['MA20'], label='20日均线', linewidth=1.5)
        ax1.set_title('总规模趋势')
        ax1.legend(loc='upper left', fontsize=8)
        ax1.grid(True, alpha=0.3)
        ax1.tick_params(axis='x', rotation=45)
    
    # 2. 大类结构
    ax2 = fig.add_subplot(gs[0, 1])
    sql_major = """
    SELECT 分类, 余额
    FROM bond.信用债规模
    WHERE DT = :dt AND 分类方式 = '大类' AND 分类 != '全部'
    ORDER BY 余额 DESC
    """
    with engine.connect() as conn:
        major = pd.read_sql(text(sql_major), conn, params={'dt': str(dt)})
    
    if len(major) > 0:
        colors = plt.cm.Set3(np.linspace(0, 1, len(major)))
        ax2.pie(major['余额'], labels=major['分类'], autopct='%1.1f%%', 
                colors=colors, startangle=90)
        ax2.set_title(f'大类结构 ({dt})')
    
    # 3. 评级分布
    ax3 = fig.add_subplot(gs[1, 0])
    sql_rating = """
    SELECT 子类 as 评级, 余额
    FROM bond.信用债规模
    WHERE DT = :dt AND 分类方式 = '评级' AND 分类 = '全部'
    ORDER BY 余额 DESC
    LIMIT 7
    """
    with engine.connect() as conn:
        rating = pd.read_sql(text(sql_rating), conn, params={'dt': str(dt)})
    
    if len(rating) > 0:
        colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(rating)))[::-1]
        ax3.bar(rating['评级'], rating['余额'], color=colors)
        ax3.set_title(f'评级分布 ({dt})')
        ax3.tick_params(axis='x', rotation=45)
    
    # 4. 久期分布
    ax4 = fig.add_subplot(gs[1, 1])
    sql_term = """
    SELECT CAST(子类 AS DECIMAL) as 久期, 余额
    FROM bond.信用债规模
    WHERE DT = :dt AND 分类方式 = '久期' AND 分类 = '全部'
    ORDER BY 久期
    """
    with engine.connect() as conn:
        term = pd.read_sql(text(sql_term), conn, params={'dt': str(dt)})
    
    if len(term) > 0:
        term['久期'] = term['久期'].astype(str) + '年'
        colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(term)))
        ax4.bar(term['久期'], term['余额'], color=colors)
        ax4.set_title(f'久期分布 ({dt})')
    
    plt.suptitle('信用债规模综合仪表板', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# 创建仪表板
create_dashboard(sql_engine, latest_date, start_date, end_date)

---

### 可视化模块完成

**已实现的可视化功能**:
1. 规模趋势图 - 日度规模与移动平均
2. 大类结构分布图 - 饼图与条形图
3. 评级分布图 - 按评级分类的规模分布
4. 久期分布图 - 按久期段分类的规模分布
5. 大类趋势对比图 - 多大类趋势对比
6. 增长率热力图 - 月度增长率热力图
7. 综合仪表板 - 多维度综合展示

---

**下一章**: [7_部署与监控](7_部署与监控.ipynb)